# Logs CPF Intergrall VF1 — Bronze Layer

**Migrated from:** Alteryx workflow `Logs CPF Intergrall VF1.yxmd`
**Migration date:** 2026-08-06
**Medallion tier:** Bronze (raw ingestion)
**Source data:**
* `bacen.dbo.cat099_log_itgl` via `odbc:DSN=SYBASE` (Sybase ASE, user `U_ITMONITOR`)
* `Base_Funcionarios_Atualizado.xlsx`, abas `Ativos$` e `DesligadosConsolidado$`
  (share SMB `\\swap629\Auditoria_Continua$`)

**Target tables:** `{bronze}.cat099_log_itgl`, `{bronze}.base_funcionarios`

## Alteryx Tool Mapping
| Step | Alteryx Tool | ToolID | Databricks Operation |
|---|---|---|---|
| 1 | Input Data (ODBC Sybase) | 9 | Tabela UC existente (leitura direta) |
| 2 | Input Data (Excel INDEX$) | 38 | `read_landing(sheet="INDEX")` → Delta bronze |

## Regras desta camada
Nomes e tipos originais preservados; nenhuma lógica de negócio. O único desvio
é o Union (Tool 15), trazido para Bronze porque as duas abas são o *mesmo*
artefato de origem — a coluna `_source_sheet` mantém a procedência.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F  # usado no read_landing + withColumn

BATCH_ID = f"{HOJE.isoformat()}T{__import__('datetime').datetime.now():%H%M%S}"

## 1. Log de consultas Intergrall (Tool 9)

A fonte de log agora é uma tabela Unity Catalog existente:
`datalake.raw_tecnologia_governanca_de_ti.bacen_cat099_log_itgl`.

Não há ingestão de arquivo — a tabela é lida diretamente. O recorte temporal
(`start_date`..`end_date`) é aplicado em Silver.

In [0]:
%sql
-- Log source is already a managed table — read directly, no ingestion needed.
-- Valida existência e exibe schema.
DESCRIBE TABLE ${log_input_table}

## 2. Base de RH — aba única `INDEX` (Base IGI RH.XLSX)

A nova base de RH tem uma única aba `INDEX` com 15 colunas:
`Nome`, `LoginAD`, `CPF`, `Matricula`, `Email`, `Status FA`, `Status IGI`,
`Data de Admissão`, `Data de Desligamento`, `Cargo`, `Departamento`, `Empresa`,
`Gestor(a)`, `Gestor(a) E-mail`, `Gestor Status`.

O status ativo/desligado é determinado pela coluna `Status FA`.

<!-- OLD: — duas abas + Union ByPos (Tools 13, 14, 15)

O Union original era **`ByPos`**: alinha por posição e ignora os nomes. As duas
abas têm as mesmas 40 colunas na mesma ordem, então funciona — mas uma coluna
inserida numa aba e não na outra desalinha tudo em silêncio. A asserção abaixo
converte esse risco latente em erro explícito.

Divergência real de tipo entre as abas: `DATA DE RESCISÃO` é `Date` em
`Ativos$` e texto em `DesligadosConsolidado$`. Como tudo é lido com
`dtype=str`, o Union não quebra; a conversão fica em Silver. -->

In [0]:
# Excel read — must stay Python (openpyxl + pandas for dtype=str preservation)
df_rh_bronze = (
    read_landing(RH_INPUT_PATH, sheet="INDEX")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(RH_INPUT_PATH))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_rh_bronze.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .saveAsTable(T_BRONZE_RH)
)

In [0]:
%sql
COMMENT ON TABLE `${catalog}`.${bronze_schema}.base_funcionarios IS
  'Bronze — base de RH (snapshot). Aba INDEX de Base IGI RH.XLSX.'

## 3. Validação de Bronze (skill, Fase 2, passo 7)

Contagens conferidas contra a origem. Estatísticas via `spark.sql()` inline
por tabela — cada `.collect()` dispara um job (item 9 de performance).

In [0]:
# Validação com assertions (precisa Python para assert + print)
log_stats = spark.sql(f"""
  SELECT
    COUNT(*)                     AS linhas,
    COUNT(DISTINCT log_itgl_seq) AS seq_distintos,
    MIN(log_itgl_dat)            AS dat_min,
    MAX(log_itgl_dat)            AS dat_max
  FROM {T_BRONZE_LOG}
""").collect()[0]

rh_stats = spark.sql(f"""
  SELECT
    COUNT(*)                     AS linhas,
    COUNT(DISTINCT Nome)         AS nomes_distintos,
    SUM(CASE WHEN upper(`Status FA`) = 'ATIVO' THEN 1 ELSE 0 END) AS ativos,
    SUM(CASE WHEN upper(`Status FA`) != 'ATIVO' THEN 1 ELSE 0 END) AS desligados,
    SUM(CASE WHEN CPF IS NULL THEN 1 ELSE 0 END) AS cpf_nulos
  FROM {T_BRONZE_RH}
""").collect()[0]

print(f"batch {BATCH_ID}")
print(f"{T_BRONZE_LOG}: {log_stats.linhas:,} linhas | "
      f"{log_stats.seq_distintos:,} seq distintos | datas {log_stats.dat_min}..{log_stats.dat_max}")
print(f"{T_BRONZE_RH}: {rh_stats.linhas:,} linhas "
      f"({rh_stats.ativos:,} ativos + {rh_stats.desligados:,} desligados) | "
      f"{rh_stats.nomes_distintos:,} nomes distintos | {rh_stats.cpf_nulos:,} CPF nulos")

assert log_stats.linhas > 0, "log Intergrall vazio — verifique a tabela fonte"
assert rh_stats.linhas > 0, "base de RH vazia — verifique o caminho do .xlsx"
assert rh_stats.ativos > 0, (
    "Nenhum funcionário com Status FA = ATIVO — verifique a base de RH"
)

In [0]:
dbutils.notebook.exit(
    f"BRONZE OK | log={log_stats.linhas} (tabela existente) | rh={rh_stats.linhas} | batch={BATCH_ID}"
)